In [1]:
import os, shutil, subprocess

In [2]:
os.makedirs("docker_1", exist_ok=True)

In [3]:
script = '''import random
random.seed(42)

SPAM_TEMPLATES = [
    "WIN a FREE {prize} now! Click here: {url}",
    "Congratulations! You have won a {prize}. Claim NOW at {url}",
    "URGENT: Your account will be suspended. Verify at {url}",
    "Limited time offer! Get {prize} FREE, click {url} today",
    "You've been selected for a {prize}! Reply YES to claim.",
    "Cash prize alert: claim your {prize} before it expires! {url}",
]

HAM_TEMPLATES = [
    "Hey, are we still meeting for {activity} on {day}?",
    "Can you send me the notes from {activity} class?",
    "Don't forget about {activity} this {day}, see you there",
    "Thanks for helping with {activity} yesterday",
    "Running a bit late for {activity}, be there in 10 min",
    "What time does {activity} start on {day}?",
]

PRIZES = ["iPhone", "cash prize", "gift card", "vacation", "laptop"]

URLS = ["bit.ly/xyz123", "tinyurl.com/abc", "win-now.co/claim"]

ACTIVITIES = ["lunch", "the study group", "basketball", "the project meeting"]

DAYS = ["Monday", "Friday", "tomorrow", "the weekend"]

rows = []

for i in range(1000):
    if random.random() < 0.3:
        t = random.choice(SPAM_TEMPLATES)
        msg = t.format(prize=random.choice(PRIZES), url=random.choice(URLS))
        rows.append((msg, "spam"))
    else:
        t = random.choice(HAM_TEMPLATES)
        msg = t.format(activity=random.choice(ACTIVITIES), day=random.choice(DAYS))
        rows.append((msg, "ham"))

import pandas as pd
pd.DataFrame(rows, columns=["text", "label"]).to_csv("spam_dataset.csv", index=False)

'''

with open("docker_1/script.py", "w") as f:
    f.write(script)

In [4]:
train_code = '''import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

df = pd.read_csv("spam_dataset.csv")

pipeline = Pipeline([("tfidf", TfidfVectorizer()), ("classifier", MultinomialNB())])
pipeline.fit(df["text"], df["label"])
joblib.dump(pipeline, "spam_predictor.joblib")
'''

with open("docker_1/train.py", "w") as f:
    f.write(train_code)

In [5]:
main_api = '''from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
import joblib

app = FastAPI()
try:
    model = joblib.load("spam_predictor.joblib")
except Exception:
    model = None

class PredictRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(input: PredictRequest):
    prediction = model.predict([input.text])[0]
    return {"label": prediction}

@app.get("/healthz", status_code = status.HTTP_200_OK)
def check_health():
    if model == None:
        raise HTTPException(status_code=503, detail="Model hasn't been loaded")
    return {"status" : "ok"}

'''

with open("docker_1/main.py", "w") as f:
    f.write(main_api)

In [6]:
requirements = '''numpy
scikit-learn
joblib
pandas
fastapi
uvicorn
'''

with open("docker_1/requirements.txt", "w") as f:
    f.write(requirements)

In [7]:
multistage_dockerfile = '''FROM python:3.10 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/build/deps -r requirements.txt

FROM python:3.10-slim AS trainer
WORKDIR /train_dir
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY script.py .
RUN python3 script.py
COPY train.py .
RUN python3 train.py

FROM python:3.10-slim
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY --from=trainer /train_dir/spam_predictor.joblib /app/spam_predictor.joblib
COPY main.py .
EXPOSE 8000
CMD ["python3", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

'''

with open("docker_1/Dockerfile.multistage", "w") as f:
    f.write(multistage_dockerfile)